# Scraper specifications

In [24]:
import json
import time
from selenium import webdriver
from selenium.webdriver.common.by import By


# ==============================================================================
# 1. CRAWL CHỈ THÔNG SỐ KỸ THUẬT
# ==============================================================================

def crawl_only_specifications(driver, product):
    url_product = "https://fptshop.com.vn" + product["product_url"]

    # Khởi tạo Schema đích ngay từ đầu cho gọn
    product_json = {
        "brand_name": product.get("brand_name"),
        "series_name": None,  # Sẽ lấy từ Breadcrumb nếu có
        "category_name": "Điện thoại",
        "product_info": {
            "name": product.get("product_name"),
            "base_name": product.get("product_name"),
            "short_description": None,
            "detail_description": None,
            "thumbnail_url": product.get("thumbnail_url"),
            "sale": None,
            "warranty_months": 12,
            "status": "ACTIVE",
            "is_featured": False
        },
        "variants": [],  # Bỏ qua không cào biến thể
        "specifications": []
    }

    try:
        driver.get(url_product)
        time.sleep(3)  # Chờ trang tải danh mục ban đầu

        # ----------------------------------------------------------
        # LẤY SERIES (TỪ BREADCRUMB)
        # ----------------------------------------------------------
        try:
            breadcrumb = driver.find_element(By.CSS_SELECTOR, "nav.Breadcrumb")
            product_json["series_name"] = (
                breadcrumb.text.split("\n")[-1]
                if "\n" in breadcrumb.text
                else None
            )
        except:
            pass

        # ----------------------------------------------------------
        # CÀO BẢNG THÔNG SỐ PHẲNG
        # ----------------------------------------------------------
        try:
            # Tìm và click nút Xem tất cả thông số
            spec_button = driver.find_element(By.XPATH, "//button[contains(., 'Xem tất cả thông số')]")
            driver.execute_script("arguments[0].click();", spec_button)
            time.sleep(2)

            # Quét các khối danh mục (Màn hình, Camera, Pin,...)
            spec_blocks = driver.find_elements(
                By.XPATH,
                "//div[contains(@class,'tab-content') and starts-with(@id,'spec-item-')]"
            )

            specs_flat_list = []
            sort_order = 1

            for block in spec_blocks:
                # Trích xuất tên nhóm danh mục lớn
                try:
                    category_name = block.find_element(
                        By.XPATH,
                        "./div[contains(@class,'b2-semibold') or contains(@class,'text-textOnWhitePrimary')]"
                    ).text.strip()
                except:
                    try:
                        category_name = block.find_element(By.XPATH, "./div[1]").text.strip()
                    except:
                        continue

                # Quét từng hàng thông số con bên trong khối
                rows = block.find_elements(By.XPATH, ".//div[contains(@class,'border-dashed') or contains(@class,'flex-row')]")
                for row in rows:
                    try:
                        key = row.find_element(By.XPATH, "./div[1]").text.strip()
                        value = row.find_element(By.XPATH, "./div[2] | ./*[2]").text.strip().replace("\n", ", ")

                        if not key and not value:
                            continue

                        # Đẩy trực tiếp thành cấu trúc phẳng map với Database của bạn
                        specs_flat_list.append({
                            "spec_category": category_name,
                            "spec_key": key,
                            "spec_value": value,
                            "sort_order": sort_order
                        })
                        sort_order += 1
                    except:
                        continue

            product_json["specifications"] = specs_flat_list
        except Exception as e:
            print(f"  [Lỗi UI] Không mở được bảng thông số của: {product.get('product_name')}")

        return product_json

    except Exception as e:
        print(f"Lỗi kết nối khi crawl {url_product}:", e)
        return None


# ==============================================================================
# 2. BỘ ĐIỀU KHIỂN CHẠY PIPELINE
# ==============================================================================

def main():
    try:
        with open("filtered_products.json", "r", encoding="utf-8") as f:
            products = json.load(f)
    except FileNotFoundError:
        print("Lỗi: Không tìm thấy file nguồn 'filtered_products.json'.")
        return

    driver = webdriver.Chrome()
    final_products = []

    print("=== BẮT ĐẦU CÀO RIÊNG THÔNG SỐ KỸ THUẬT ===")
    try:
        for index, product in enumerate(products, start=1):
            print(f"[{index}/{len(products)}] Đang cào: {product.get('product_name')}")

            # Gọi hàm cào trực tiếp trả về định dạng đích (Không cần qua hàm Transform trung gian nữa)
            result = crawl_only_specifications(driver, product)
            
            if result:
                final_products.append(result)

            # Auto-save lưu tiến trình liên tục đề phòng rớt mạng giữa chừng
            if index % 10 == 0:
                with open("products_spes.json", "w", encoding="utf-8") as f:
                    json.dump(final_products, f, ensure_ascii=False, indent=2)
                print("  [Checkpoint]: Đã lưu dữ liệu dự phòng.")

        # Lưu file thành phẩm cuối cùng
        with open("products_spes.json", "w", encoding="utf-8") as f:
            json.dump(final_products, f, ensure_ascii=False, indent=2)

        print(f"\n=== HOÀN THÀNH: Đã xuất thành công {len(final_products)} sản phẩm chỉ chứa thông số! ===")

    finally:
        driver.quit()


if __name__ == "__main__":
    main()

=== BẮT ĐẦU CÀO RIÊNG THÔNG SỐ KỸ THUẬT ===
[1/60] Đang cào: Xiaomi 17T 5G 12GB 512GB
  [Lỗi UI] Không mở được bảng thông số của: Xiaomi 17T 5G 12GB 512GB
[2/60] Đang cào: Xiaomi 17T Pro 5G 12GB 512GB
  [Lỗi UI] Không mở được bảng thông số của: Xiaomi 17T Pro 5G 12GB 512GB
[3/60] Đang cào: iPhone 17 256GB
[4/60] Đang cào: Honor X9d 5G 8GB 256GB
[5/60] Đang cào: Samsung Galaxy S25 Ultra 5G 12GB 256GB
[6/60] Đang cào: REDMAGIC 11 Pro 5G 12GB 256GB
[7/60] Đang cào: Xiaomi Poco X7 5G 12GB 512GB
[8/60] Đang cào: Samsung Galaxy A17 8GB 128GB
[9/60] Đang cào: Honor X7d 8GB 128GB
[10/60] Đang cào: iPhone 17 Pro 256GB
  [Checkpoint]: Đã lưu dữ liệu dự phòng.
[11/60] Đang cào: Xiaomi Redmi 13x 8GB 128GB
[12/60] Đang cào: Samsung Galaxy A07 4GB 128GB
[13/60] Đang cào: Samsung Galaxy S25 FE 5G 8GB 128GB
[14/60] Đang cào: Honor X6c 6GB 128GB
[15/60] Đang cào: Nubia A76 4GB 128GB (NFC)
[16/60] Đang cào: Xiaomi Poco M7 Pro 5G 8GB 256GB
[17/60] Đang cào: Nubia V70 Design 8GB 128GB
[18/60] Đang cào: Ma

# Kiểm tra các product thietes specifications

In [3]:
import json

with open("products_data_final_merged.json", "r", encoding="utf-8") as f:
    products = json.load(f)

cnt = sum(1 for product in products if not product.get("specifications"))
print(f"Số sản phẩm thiếu specifications: {cnt}")
for product in products:
    if not product.get("specifications"):
        print(f"Sản phẩm thiếu specifications: {product.get('product_info', {}).get('name')}")

count = sum(
    1
    for product in products
    if product.get("specifications")
)
products_non_spes = [product.get("product_info", {}).get("name") for product in products if not product.get("specifications")]

print(f"Số sản phẩm có specifications: {count}")
print(f"Tổng số sản phẩm: {len(products)}")

Số sản phẩm thiếu specifications: 8
Sản phẩm thiếu specifications: Xiaomi 17T 5G
Sản phẩm thiếu specifications: Xiaomi 17T Pro 5G
Sản phẩm thiếu specifications: iPhone 17 Pro Max
Sản phẩm thiếu specifications: Samsung Galaxy Z Fold7 5G
Sản phẩm thiếu specifications: OPPO Reno15 F 5G
Sản phẩm thiếu specifications: Nubia Neo 5 5G
Sản phẩm thiếu specifications: Samsung Galaxy Z TriFold 5G
Sản phẩm thiếu specifications: REDMAGIC 11s Pro 5G
Số sản phẩm có specifications: 146
Tổng số sản phẩm: 154


In [4]:
print("Sản phẩm thiếu specifications:")
for name in products_non_spes:
    print(f"  - {name}")

Sản phẩm thiếu specifications:
  - Xiaomi 17T 5G
  - Xiaomi 17T Pro 5G
  - iPhone 17 Pro Max
  - Samsung Galaxy Z Fold7 5G
  - OPPO Reno15 F 5G
  - Nubia Neo 5 5G
  - Samsung Galaxy Z TriFold 5G
  - REDMAGIC 11s Pro 5G


In [5]:
import json

def filter_products_by_names(products_file, output_file, name_list):
    # Đọc dữ liệu từ file products.json của bạn
    try:
        with open(products_file, 'r', encoding='utf-8') as f:
            products_data = json.load(f)
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file '{products_file}'. Hãy kiểm tra lại đường dẫn.")
        return

    # Đảm bảo dữ liệu đầu vào là một list
    if isinstance(products_data, dict):
        products_data = [products_data]

    filtered_result = []

    # Duyệt qua từng sản phẩm trong products.json
    for product in products_data:
        p_name = product.get("product_name", "")
        if not p_name:
            continue
            
        # Kiểm tra xem tên sản phẩm trong file có chứa bất kỳ tên nào trong danh sách lọc không
        # Dùng .lower() để tránh việc lệch chữ hoa / chữ thường (ví dụ: iPhone vs iphone)
        for target_name in name_list:
            if target_name.strip().lower() in p_name.lower():
                filtered_result.append(product)
                break # Nếu khớp rồi thì dừng vòng lặp target_name để chuyển sang sản phẩm tiếp theo
    print(f"Đã lọc xong {len(filtered_result)} sản phẩm khớp với danh sách tên cần tìm.")
    # Ghi kết quả ra file mới
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(filtered_result, f, ensure_ascii=False, indent=4)

    print(f"😎 Đã lọc xong!")
    print(f"🔍 Tìm thấy {len(filtered_result)} sản phẩm khớp trong file '{products_file}'.")
    print(f"💾 Đã lưu kết quả tại file: '{output_file}'")

# ==============================================================================
# DANH SÁCH TÊN SẢN PHẨM CẦN LỌC (Copy từ danh sách của bạn)
# ==============================================================================
# Chạy hàm thực thi (Đọc từ 'products.json' -> Xuất ra 'filtered_products.json')
filter_products_by_names('products.json', 'filtered_products.json', products_non_spes)

Đã lọc xong 8 sản phẩm khớp với danh sách tên cần tìm.
😎 Đã lọc xong!
🔍 Tìm thấy 8 sản phẩm khớp trong file 'products.json'.
💾 Đã lưu kết quả tại file: 'filtered_products.json'


# Gộp dữ liệu vào products_data_final.json

In [2]:
import json
import os

def merge_specifications(base_file, specs_file, output_file):
    # 1. Kiểm tra sự tồn tại của các file nguồn
    if not os.path.exists(base_file) or not os.path.exists(specs_file):
        print("❌ Lỗi: Không tìm thấy file 'products_data_final.json' hoặc 'products_spes.json'.")
        return

    # 2. Đọc dữ liệu từ 2 file JSON
    with open(base_file, 'r', encoding='utf-8') as f:
        base_data = json.load(f)
    with open(specs_file, 'r', encoding='utf-8') as f:
        specs_data = json.load(f)

    # Đảm bảo dữ liệu luôn ở dạng list để duyệt loop
    if isinstance(base_data, dict): base_data = [base_data]
    if isinstance(specs_data, dict): specs_data = [specs_data]

    success_count = 0

    # 3. Tiến hành duyệt và map dữ liệu theo logic yêu cầu
    for base_item in base_data:
        base_info = base_item.get("product_info", {})
        base_name = base_info.get("name") if base_info else None
        base_specs = base_item.get("specifications", [])

        # Chỉ xử lý nếu sản phẩm ở file gốc đang KHÔNG có specifications (hoặc rỗng)
        if base_name and (base_specs is None or len(base_specs) == 0):
            
            # Chạy vòng lặp quét qua file dữ liệu đã cào để tìm sản phẩm khớp tên
            for spec_item in specs_data:
                spec_info = spec_item.get("product_info", {})
                spec_name = spec_info.get("name") if spec_info else None
                
                # Điều kiện 1: Tên trong products_spes.json CHỨA tên trong products_data_final.json
                if spec_name and (base_name.lower() in spec_name.lower()):
                    
                    # Đổ mảng specifications từ file spes sang file gốc
                    base_item["specifications"] = spec_item.get("specifications", [])
                    
                    # Cập nhật thêm cả series_name nếu file gốc đang trống mà file cào được lại có
                    if not base_item.get("series_name") and spec_item.get("series_name"):
                        base_item["series_name"] = spec_item.get("series_name")
                        
                    success_count += 1
                    break # Đã tìm thấy sản phẩm khớp phù hợp, chuyển sang sản phẩm tiếp theo ở file gốc

    # 4. Ghi kết quả sau khi hợp nhất ra file thành phẩm mới
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(base_data, f, ensure_ascii=False, indent=2)

    print("=========================================================")
    print(f"😎 Hợp nhất dữ liệu hoàn tất!")
    print(f"🔄 Đã đổ thành công specifications cho {success_count} sản phẩm.")
    print(f"💾 File mới chứa dữ liệu trọn vẹn lưu tại: '{output_file}'")
    print("=========================================================")

# --- THỰC THI CELL CODE ---
merge_specifications(
    base_file='products_data_final.json', 
    specs_file='products_spes.json', 
    output_file='products_data_final_merged.json'
)

😎 Hợp nhất dữ liệu hoàn tất!
🔄 Đã đổ thành công specifications cho 55 sản phẩm.
💾 File mới chứa dữ liệu trọn vẹn lưu tại: 'products_data_final_merged.json'


# Xóa các sản phẩm thiếu specifications 

In [6]:
def remove_empty_specs_products(input_file, output_file):
    # 1. Kiểm tra file nguồn
    if not os.path.exists(input_file):
        print(f"❌ Lỗi: Không tìm thấy file '{input_file}'.")
        return

    # 2. Đọc dữ liệu từ file merged
    with open(input_file, 'r', encoding='utf-8') as f:
        products_data = json.load(f)

    if isinstance(products_data, dict):
        products_data = [products_data]

    initial_count = len(products_data)
    
    # 3. Lọc: Chỉ giữ lại các sản phẩm có specifications và mảng specifications không rỗng
    cleaned_products = []
    for item in products_data:
        specs = item.get("specifications")
        
        # Điều kiện giữ lại: specs phải tồn tại và có độ dài > 0
        if specs is not None and isinstance(specs, list) and len(specs) > 0:
            cleaned_products.append(item)

    final_count = len(cleaned_products)
    removed_count = initial_count - final_count

    # 4. Ghi dữ liệu đã làm sạch ra file mới
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(cleaned_products, f, ensure_ascii=False, indent=2)

    print("=========================================================")
    print(f"✨ ĐÃ LÀM SẠCH DATABASE THÀNH CÔNG!")
    print(f"📊 Tổng số sản phẩm ban đầu: {initial_count}")
    print(f"🗑️ Đã xóa bỏ: {removed_count} sản phẩm không có thông số kỹ thuật.")
    print(f"✅ Số sản phẩm hợp lệ còn lại: {final_count}")
    print(f"💾 Kết quả cuối cùng được lưu tại: '{output_file}'")
    print("=========================================================")

# --- THỰC THI CELL CODE ---
# Xuất trực tiếp ra file thành phẩm chuẩn cuối cùng của dự án
remove_empty_specs_products(
    input_file='products_data_final_merged.json',
    output_file='products_data_final_cleaned.json'
)

✨ ĐÃ LÀM SẠCH DATABASE THÀNH CÔNG!
📊 Tổng số sản phẩm ban đầu: 154
🗑️ Đã xóa bỏ: 8 sản phẩm không có thông số kỹ thuật.
✅ Số sản phẩm hợp lệ còn lại: 146
💾 Kết quả cuối cùng được lưu tại: 'products_data_final_cleaned.json'


## Lọc bỏ button

In [4]:
import json
import os
from bs4 import BeautifulSoup

def clean_html_description(input_file, output_file):
    if not os.path.exists(input_file):
        print(f"❌ Lỗi: Không tìm thấy file nguồn '{input_file}'")
        return

    # 1. Đọc dữ liệu từ file JSON sạch
    with open(input_file, 'r', encoding='utf-8') as f:
        products = json.load(f)

    if isinstance(products, dict):
        products = [products]

    cleaned_count = 0

    # 2. Quét qua từng sản phẩm để xử lý HTML
    for item in products:
        product_info = item.get("product_info", {})
        html_content = product_info.get("detail_description")

        if html_content and isinstance(html_content, str):
            soup = BeautifulSoup(html_content, 'html.parser')

            # 1. XÓA NÚT ĐỌC THÊM VÀ LỚP PHỦ GRADIENT (Code cũ của bạn)
            read_more_block = soup.find('div', class_=lambda c: c and 'linear-gradient' in c)
            if read_more_block:
                read_more_block.decompose()

            # 2. SỬA LỖI GIAO DIỆN: Tìm div bị bóp max-height để xóa bỏ giới hạn chiều cao
            bounded_div = soup.find('div', style=lambda s: s and 'max-height' in s)
            if bounded_div:
                # Xóa bỏ thuộc tính style="max-height: 499px;"
                del bounded_div['style']

            # Cập nhật lại HTML sạch hoàn toàn vào cấu trúc
            product_info["detail_description"] = str(soup)
            cleaned_count += 1

    # 4. Lưu lại dữ liệu chuẩn sau khi lọc cấu trúc HTML
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(products, f, ensure_ascii=False, indent=2)

    print("=========================================================")
    print(f"✨ LÀM SẠCH MÔ TẢ HTML HOÀN TẤT!")
    print(f"✂️ Đã xử lý cắt bỏ UI thừa cho {cleaned_count} bài viết chi tiết.")
    print(f"💾 File JSON kết quả lưu tại: '{output_file}'")
    print("=========================================================")

# --- CHẠY THỰC THI CELL CODE ---
# Lưu đè lên chính file ready để đồng bộ dữ liệu chuẩn bị import
clean_html_description(
    input_file='products_data_final_cleaned.json',
    output_file='products_data_final_cleaned.json'
)

✨ LÀM SẠCH MÔ TẢ HTML HOÀN TẤT!
✂️ Đã xử lý cắt bỏ UI thừa cho 130 bài viết chi tiết.
💾 File JSON kết quả lưu tại: 'products_data_final_cleaned.json'


## Lọc bỏ cost = 0

In [5]:
import json
import os

def remove_products_with_zero_cost_price(input_file, output_file):
    if not os.path.exists(input_file):
        print(f"❌ Lỗi: Không tìm thấy file nguồn '{input_file}'")
        return

    # 1. Đọc dữ liệu từ file JSON hiện tại
    with open(input_file, 'r', encoding='utf-8') as f:
        products = json.load(f)

    if isinstance(products, dict):
        products = [products]

    initial_count = len(products)
    cleaned_products = []

    # 2. Duyệt qua từng sản phẩm để kiểm tra mảng variants
    for item in products:
        variants = item.get("variants", []) or []
        
        # Biến cờ đánh dấu xem sản phẩm này có bị dính lỗi cost_price = 0 hay không
        has_zero_cost_price = False
        
        for variant in variants:
            # Kiểm tra nếu cost_price tồn tại và bằng chính xác 0
            if variant.get("cost_price") == 0:
                has_zero_cost_price = True
                break # Phát hiện một variant lỗi là đủ để loại bỏ sản phẩm này
                
        # Nếu không có biến thể nào có giá vốn bằng 0, giữ sản phẩm lại
        if not has_zero_cost_price:
            cleaned_products.append(item)

    final_count = len(cleaned_products)
    removed_count = initial_count - final_count

    # 3. Ghi lại dữ liệu sạch ra file
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(cleaned_products, f, ensure_ascii=False, indent=2)

    print("=========================================================")
    print(f"🗑️ Đã lọc bỏ {removed_count} sản phẩm có cost_price bằng 0.")
    print(f"✅ Số sản phẩm hợp lệ còn lại: {final_count} / {initial_count}")
    print(f"💾 File dữ liệu mới đã được lưu tại: '{output_file}'")
    print("=========================================================")

# --- CHẠY THỰC THI CELL CODE ---
# Cập nhật trực tiếp vào file chuẩn bị import của bạn
remove_products_with_zero_cost_price(
    input_file='products_data_final_cleaned.json',
    output_file='products_data_final_cleaned.json'
)

🗑️ Đã lọc bỏ 0 sản phẩm có cost_price bằng 0.
✅ Số sản phẩm hợp lệ còn lại: 130 / 130
💾 File dữ liệu mới đã được lưu tại: 'products_data_final_cleaned.json'


## Lọc bỏ variants rỗng 

In [6]:
import json
import os

def remove_empty_variants_products(input_file, output_file):
    if not os.path.exists(input_file):
        print(f"❌ Lỗi: Không tìm thấy file nguồn '{input_file}'")
        return

    # 1. Đọc dữ liệu từ file JSON hiện tại
    with open(input_file, 'r', encoding='utf-8') as f:
        products = json.load(f)

    if isinstance(products, dict):
        products = [products]

    initial_count = len(products)
    cleaned_products = []

    # 2. Duyệt qua từng sản phẩm để kiểm tra mảng variants
    for item in products:
        variants = item.get("variants")
        
        # Điều kiện giữ lại: variants phải tồn tại, phải là dạng list và có số lượng phần tử > 0
        if variants is not None and isinstance(variants, list) and len(variants) > 0:
            cleaned_products.append(item)

    final_count = len(cleaned_products)
    removed_count = initial_count - final_count

    # 3. Ghi lại dữ liệu sạch ra file thành phẩm
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(cleaned_products, f, ensure_ascii=False, indent=2)

    print("=========================================================")
    print(f"🗑️ Đã lọc bỏ: {removed_count} sản phẩm có mảng variants rỗng.")
    print(f"✅ Số sản phẩm hợp lệ còn lại: {final_count} / {initial_count}")
    print(f"💾 File dữ liệu mới đã được lưu tại: '{output_file}'")
    print("=========================================================")

# --- CHẠY THỰC THI CELL CODE ---
# Cập nhật trực tiếp vào file chuẩn bị import của bạn để đồng bộ
remove_empty_variants_products(
    input_file='products_data_final_cleaned.json',
    output_file='products_data_final_cleaned.json'
)

🗑️ Đã lọc bỏ: 0 sản phẩm có mảng variants rỗng.
✅ Số sản phẩm hợp lệ còn lại: 130 / 130
💾 File dữ liệu mới đã được lưu tại: 'products_data_final_cleaned.json'
